# BERT Similarity Experiments

Тот же фича-инжиниринг, что и в ML_Experiments.ipynb, включая ALS.
**Cosine similarity вычислены через BERT-модели вместо TF-IDF.**

In [1]:
import os
import nltk
import pymorphy3
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import mlflow
import optuna

from tqdm.auto import tqdm
from scipy.sparse import csr_matrix, vstack
from transformers import AutoTokenizer, AutoModel
from implicit.als import AlternatingLeastSquares
from optuna.integration.mlflow import MLflowCallback
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from dotenv import load_dotenv
from clickhouse_driver import Client
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess

warnings.simplefilter('ignore', FutureWarning)


In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000
EXPERIMENT_NAME = "hr-ai-scout-bert"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")


In [3]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Device: {DEVICE}")


Device: mps


## 1. Load data

In [4]:
df = pd.read_csv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/total_df.csv')
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_salary_from,vacancy_salary_to,vacancy_salary_currency,vacancy_salary_gross,...,resume_education,resume_courses,resume_salary,resume_age,resume_total_experience,resume_experience_months,resume_location,resume_gender,resume_applicant_status,target
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Казанский Авиационный Институт'],NaN,NaN,65.0,19 лет,228.0,Москва,Мужчина,Рассматривает предложения,1
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,"['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...","['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...",NaN,43.0,17 лет 4 месяца,208.0,Москва,Мужчина,Рассматривает предложения,1
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Орский государственный педагогический инстит...,NaN,200 000 ₽ на руки,52.0,30 лет,360.0,Москва,Женщина,NaN,1
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Красноярский государственный университет'],NaN,500 000 ₽ на руки,56.0,29 лет 8 месяцев,356.0,Красноярск,Мужчина,Рассматривает предложения,1
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Белоруский Гос. Университет Информатики и Ра...,"['SAP CIS, SAP XI', 'Школа Логистики МАДИ', 'S...",NaN,48.0,25 лет 1 месяц,301.0,Moscow,Male,NaN,1


## 2. Preprocessing

*(скопировано из ML_Experiments.ipynb без изменений)*

In [5]:
t1 = df.shape[0]
df = df.dropna(subset=[
    "resume_education", "resume_last_experience_description",
    "resume_last_position", "resume_last_company_experience_period",
    "resume_total_experience", "resume_experience_months",
    "resume_location", "resume_specialization",
], how="all")
print('Удалено ', t1 - df.shape[0], ' строки')


Удалено  84  строки


In [6]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].notna()
              & df["resume_last_experience_description"].isna()
              & df["resume_last_position"].isna())]
df = df.loc[~(df["resume_total_experience"].isna()
              & df["resume_last_experience_description"].notna()
              & df["resume_last_position"].notna())]
print('Удалено ', t1 - df.shape[0], ' строк')


Удалено  1543  строк


In [7]:
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna('NDT')
df['resume_age'] = df['resume_age'].fillna(df['resume_age'].mean())
df['resume_experience_months'] = df['resume_experience_months'].fillna(0)
df = df.drop(['vacancy_salary_to', 'vacancy_salary_from',
              'vacancy_salary_currency', 'vacancy_salary_gross'], axis=1)


In [8]:
df['resume_salary_split'] = df['resume_salary'].apply(lambda x: x.split())
df['salary_int'] = df['resume_salary_split'].apply(
    lambda x: int(''.join(p for p in x if re.fullmatch(r'\d+', p)))
              if any(re.fullmatch(r'\d+', p) for p in x) else np.nan
)
currency_symbols = ['₽', '$', '€', '₴', '₸', '₼', '₾', 'Br', "so'm"]
rates_rub = {'₽':1.0,'$':80.85,'€':94.14,'₴':1.94,'₸':0.150,
             '₼':47.8,'₾':33.5,'Br':28.7,"so'm":0.0068}
df['currency_symbol'] = df['resume_salary_split'].apply(
    lambda x: next((s for s in x if s in currency_symbols), np.nan))
df['salary_converted'] = (df['salary_int'] * df['currency_symbol'].map(rates_rub)).fillna(0)
df['resume_salary'] = df['salary_converted']
df = df.drop(['resume_salary_split','salary_int','currency_symbol','salary_converted'], axis=1)


In [9]:
def experience_to_months(text):
    months = 0
    for pat in [r'(\d+)\s*год', r'(\d+)\s*лет']:
        m = re.search(pat, text)
        if m: months += int(m.group(1)) * 12
    m = re.search(r'(\d+)\s*месяц', text)
    if m: months += int(m.group(1))
    return months if months > 0 else np.nan

df['resume_last_company_experience_months'] = \
    df['resume_last_company_experience_period'].apply(experience_to_months)
df['resume_last_company_experience_months'] = \
    df['resume_last_company_experience_months'].fillna(0)


In [10]:
df = df[~(df.resume_salary > 1e7)]
df.loc[df['resume_experience_months'] > 720, 'resume_experience_months'] = 720
df.loc[df['resume_last_company_experience_months'] > 720, 'resume_last_company_experience_months'] = 720
df = df[~(df.resume_age > 90)]
df = df[~(df.resume_experience_months < df.resume_last_company_experience_months)]
df = df[~(df.resume_age < (df.resume_experience_months // 12) + 16)]

gender_map = {'Мужчина':'Мужчина','Male':'Мужчина','Женщина':'Женщина','Female':'Женщина'}
df['resume_gender'] = df['resume_gender'].apply(
    lambda x: gender_map.get(x, 'Неизвестно'))

print(f"Датасет после очистки: {df.shape}")


Датасет после очистки: (325543, 25)


## 3. Feature engineering (без TF-IDF)

In [11]:
# Признак совпадения локации
df['location_matching'] = (df['vacancy_area'] == df['resume_location']).astype(int)

# Количество навыков резюме в тексте вакансии
def resume_skill_count_in_vacancy(row):
    skills = row['resume_skills'].replace('[','').replace(']','').replace("'","").split(', ')
    return sum(1 for s in skills if s in row['vacancy_description'])

df['resume_skill_count_in_vacancy'] = df.apply(resume_skill_count_in_vacancy, axis=1)

# Доля слов последней должности, встречающихся в описании вакансии
def last_position_in_vacancy(row):
    bow = []
    for sep in [' ', '-', '_']:
        bow += row['resume_last_position'].split(sep=sep)
    bow = list(set(bow))
    return sum(1 for w in bow if w in row['vacancy_description']) / len(bow)

df['last_position_in_vacancy'] = df.apply(last_position_in_vacancy, axis=1)

print("Признаки добавлены: location_matching, resume_skill_count_in_vacancy, last_position_in_vacancy")


Признаки добавлены: location_matching, resume_skill_count_in_vacancy, last_position_in_vacancy


## 4. BERT Similarity

Вместо TF-IDF считаем эмбеддинги через `AutoTokenizer + AutoModel`.
Ключ оптимизации — кодируем только **уникальные** вакансии и резюме, затем маппим на все строки df.

In [12]:
def encode_texts(texts, tokenizer, model, batch_size=64, prefix=''):
    """
    Батчевое кодирование текстов в L2-нормированные эмбеддинги.
    Mean pooling по токенам, взвешенный attention mask.
    """
    if prefix:
        texts = [prefix + t for t in texts]

    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="    encoding"):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            out = model(**encoded)

        # Mean pooling
        token_emb = out.last_hidden_state                              # (B, T, H)
        mask = encoded['attention_mask'].unsqueeze(-1).float()         # (B, T, 1)
        pooled = (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        pooled = F.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)


In [13]:
def compute_bert_similarity(df, tokenizer, model, batch_size=64,
                             vacancy_prefix='', resume_prefix=''):
    """
    Cosine similarity между vacancy_description и resume_last_experience_description.
    Эмбеддинги вычисляются только для уникальных текстов.
    """
    df = df.copy()
    df['vacancy_description'] = df['vacancy_description'].fillna('').astype(str)
    df['resume_last_experience_description'] = \
        df['resume_last_experience_description'].fillna('').astype(str)

    # ── Уникальные вакансии ──────────────────────────────────────────
    unique_vac = df[['vacancy_id','vacancy_description']].drop_duplicates('vacancy_id')
    print(f"  Уникальных вакансий: {len(unique_vac)} / всего строк: {len(df)}")
    print("  Эмбеддинги вакансий...")
    vac_emb = encode_texts(unique_vac['vacancy_description'].tolist(),
                           tokenizer, model, batch_size, prefix=vacancy_prefix)
    vac_map = dict(zip(unique_vac['vacancy_id'], vac_emb))

    # ── Уникальные резюме ────────────────────────────────────────────
    unique_res = df[['resume_id','resume_last_experience_description']].drop_duplicates('resume_id')
    print(f"  Уникальных резюме: {len(unique_res)}")
    print("  Эмбеддинги резюме...")
    res_emb = encode_texts(unique_res['resume_last_experience_description'].tolist(),
                           tokenizer, model, batch_size, prefix=resume_prefix)
    res_map = dict(zip(unique_res['resume_id'], res_emb))

    # ── Попарное сходство (L2-норм → cosine = dot) ───────────────────
    sims = []
    for _, row in df.iterrows():
        v = vac_map.get(row['vacancy_id'])
        r = res_map.get(row['resume_id'])
        sims.append(float(np.dot(v, r)) if v is not None and r is not None else 0.0)

    return sims


In [14]:
BERT_MODELS = [
    ('cointegrated/LaBSE-en-ru', '', '', 64),
    ('ai-forever/sbert_large_nlu_ru', '', '', 32),
    ('intfloat/multilingual-e5-base', 'query: ', 'passage: ', 32),
]

## 4.1 Кеш эмбеддингов в ClickHouse

In [15]:
load_dotenv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/.env')

ch = Client(
    host=os.getenv('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.getenv('CLICKHOUSE_PORT', 9000)),
    user=os.getenv('CLICKHOUSE_USER', 'default'),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
    database=os.getenv('CLICKHOUSE_DATABASE', 'default'),
)

In [16]:
def get_missing_ids(ids_needed: list, table: str, id_col: str,
                    model_name: str, clickhouse) -> list:
    """
    Возвращает те id из ids_needed, для которых в ClickHouse
    ещё нет эмбеддингов (по model_name).
    """
    if not ids_needed:
        return []
    str_ids = [str(i) for i in ids_needed]
    rows = clickhouse.execute(
        f"SELECT {id_col} FROM {table} "
        f"WHERE model_name = %(m)s AND {id_col} IN %(ids)s",
        {'m': model_name, 'ids': str_ids}
    )
    existing = {row[0] for row in rows}
    missing = [i for i in str_ids if i not in existing]
    print(f"  {table}: {len(existing)} в кеше, {len(missing)} новых из {len(str_ids)}")
    return missing


def save_embeddings_to_ch(emb_map: dict, id_col: str, table: str,
                           model_name: str, clickhouse):
    """Дописывает только новые эмбеддинги — существующие не удаляет."""
    rows = [
        (str(k), model_name, v.tolist())
        for k, v in emb_map.items()
    ]
    clickhouse.execute(
        f"INSERT INTO {table} ({id_col}, model_name, embedding) VALUES",
        rows,
    )
    print(f"  Сохранено {len(rows)} эмбеддингов → {table}")


def load_embeddings_from_ch(table: str, id_col: str, model_name: str,
                              clickhouse, ids: list = None) -> dict:
    """
    Загружает эмбеддинги из ClickHouse.
    ids — если передан, загружает только указанные id (экономит память).
    """
    params = {'m': model_name}
    query = f"SELECT {id_col}, embedding FROM {table} WHERE model_name = %(m)s"
    if ids:
        params['ids'] = [str(i) for i in ids]
        query += f" AND {id_col} IN %(ids)s"
    rows = clickhouse.execute(query, params)
    return {row[0]: np.array(row[1], dtype=np.float32) for row in rows}


In [17]:
bert_sim_cols = {}

for model_name, vac_prefix, res_prefix, bs in BERT_MODELS:
    short = model_name.split('/')[-1].replace('-', '_').lower()
    col   = f'sim_{short}'
    print(f"\n{'='*60}\n{model_name}\n{'='*60}")

    # Уникальные тексты текущего датасета
    unique_vac = (df[['vacancy_id', 'vacancy_description']]
                  .drop_duplicates('vacancy_id'))
    unique_res = (df[['resume_id', 'resume_last_experience_description']]
                  .drop_duplicates('resume_id'))

    all_vac_ids = unique_vac['vacancy_id'].tolist()
    all_res_ids = unique_res['resume_id'].tolist()

    # Проверяем, каких id нет в кеше
    missing_vac = get_missing_ids(all_vac_ids, 'vacancy_embeddings',
                                   'vacancy_id', model_name, ch)
    missing_res = get_missing_ids(all_res_ids, 'resume_embeddings',
                                   'resume_id', model_name, ch)

    # Загружаем BERT только если есть пропуски
    if missing_vac or missing_res:
        tokenizer  = AutoTokenizer.from_pretrained(model_name)
        bert_model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

        if missing_vac:
            texts = (unique_vac[unique_vac['vacancy_id'].astype(str).isin(missing_vac)]
                     ['vacancy_description'].fillna('').tolist())
            emb = encode_texts(texts, tokenizer, bert_model, bs, prefix=vac_prefix)
            save_embeddings_to_ch(dict(zip(missing_vac, emb)),
                                  'vacancy_id', 'vacancy_embeddings', model_name, ch)

        if missing_res:
            texts = (unique_res[unique_res['resume_id'].astype(str).isin(missing_res)]
                     ['resume_last_experience_description'].fillna('').tolist())
            emb = encode_texts(texts, tokenizer, bert_model, bs, prefix=res_prefix)
            save_embeddings_to_ch(dict(zip(missing_res, emb)),
                                  'resume_id', 'resume_embeddings', model_name, ch)

        del bert_model, tokenizer
        if DEVICE.type == 'mps':    torch.mps.empty_cache()
        elif DEVICE.type == 'cuda': torch.cuda.empty_cache()
    else:
        print("  Все эмбеддинги уже в кеше ClickHouse — загружаем")

    # Загружаем только нужные id текущего датасета
    vac_map = load_embeddings_from_ch('vacancy_embeddings', 'vacancy_id',
                                       model_name, ch, ids=all_vac_ids)
    res_map = load_embeddings_from_ch('resume_embeddings', 'resume_id',
                                       model_name, ch, ids=all_res_ids)

    # Косинусное сходство для каждой строки df
    df['vacancy_description'] = df['vacancy_description'].fillna('').astype(str)
    df['resume_last_experience_description'] = \
        df['resume_last_experience_description'].fillna('').astype(str)

    sims = [
        float(np.dot(vac_map[str(row.vacancy_id)], res_map[str(row.resume_id)]))
        if str(row.vacancy_id) in vac_map and str(row.resume_id) in res_map
        else 0.0
        for row in df.itertuples()
    ]
    df[col] = sims
    bert_sim_cols[model_name] = col
    print(f"  Среднее сходство: {df[col].mean():.4f}")

print("\nГотово:", list(bert_sim_cols.values()))



cointegrated/LaBSE-en-ru
  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20391 в кеше, 0 новых из 20391
  Все эмбеддинги уже в кеше ClickHouse — загружаем
  Среднее сходство: 0.6485

ai-forever/sbert_large_nlu_ru
  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20391 в кеше, 0 новых из 20391
  Все эмбеддинги уже в кеше ClickHouse — загружаем
  Среднее сходство: 0.6444

intfloat/multilingual-e5-base
  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20391 в кеше, 0 новых из 20391
  Все эмбеддинги уже в кеше ClickHouse — загружаем
  Среднее сходство: 0.8039

Готово: ['sim_labse_en_ru', 'sim_sbert_large_nlu_ru', 'sim_multilingual_e5_base']


## 5. ALS

*(скопировано из ML_Experiments.ipynb без изменений)*

In [18]:
def create_interaction_matrix(df):
    unique_vacancies = df['vacancy_id'].unique().tolist()
    unique_resumes   = df['resume_id'].unique().tolist()
    id2vacancy = dict(enumerate(unique_vacancies))
    id2resume  = dict(enumerate(unique_resumes))
    vacancy2id = {v: k for k, v in id2vacancy.items()}
    resume2id  = {v: k for k, v in id2resume.items()}
    rows = [vacancy2id[v] for v in df['vacancy_id']]
    cols = [resume2id[r]  for r in df['resume_id']]
    matrix = csr_matrix((df['target'], (rows, cols)),
                        shape=(len(unique_vacancies), len(unique_resumes)),
                        dtype=np.float32)
    return matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes

def get_factors(interactions_matrix):
    als = AlternatingLeastSquares(
        factors=64, regularization=0.1, iterations=30,
        random_state=RANDOM_STATE, num_threads=0)
    als.fit(interactions_matrix.T)
    return als.item_factors, als.user_factors

def get_als_score(vacancy_id, resume_id):
    if vacancy_id not in vacancy2id or resume_id not in resume2id:
        return 0
    return float(np.dot(vacancy_factors[vacancy2id[vacancy_id]],
                         resume_factors[resume2id[resume_id]]))


## 6. Train / Test split + ALS score

In [19]:
# Базовые признаки (без similarity — добавим bert-вариант в цикле)
BASE_FEATURES = [
    'vacancy_area', 
    'vacancy_experience', 
    'vacancy_employment', 
    'vacancy_schedule',
    'resume_salary', 
    'resume_age', 
    'resume_experience_months', 
    'resume_location',
    'resume_gender', 
    'resume_applicant_status', 
    'resume_last_company_experience_months',
    'location_matching', 
    'resume_skill_count_in_vacancy', 
    'last_position_in_vacancy',
]

categorical_features = df[BASE_FEATURES].select_dtypes(exclude=np.number).columns.tolist()
numeric_features     = df[BASE_FEATURES].select_dtypes(include=np.number).columns.tolist()

X_base = df[BASE_FEATURES].copy()
y      = df['target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (260434, 14), Test: (65109, 14)


In [20]:
# ALS: обучаем на части train, чтобы избежать data leakage
X_train_als, _, y_train_als, _ = train_test_split(
    X_train, y_train, test_size=0.3, random_state=RANDOM_STATE, stratify=y_train)

train_als_interactions = df.loc[X_train_als.index, ['vacancy_id','resume_id','target']]
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, \
    unique_vacancies, unique_resumes = create_interaction_matrix(train_als_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

X_train['als_score'] = df.loc[X_train.index].apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), axis=1)

# Для test — ALS на полном train
train_interactions = df.loc[X_train.index, ['vacancy_id','resume_id','target']]
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, \
    unique_vacancies, unique_resumes = create_interaction_matrix(train_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

X_test['als_score'] = df.loc[X_test.index].apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), axis=1)

print(f"als_score в train (нули): {(X_train['als_score']==0).sum()}")
print(f"als_score в test  (нули): {(X_test['als_score']==0).sum()}")


/Users/user/Documents/Magistracy/year_project/hr-ai-scout/venv_hr_ai_scout/lib/python3.13/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.00041103363037109375 seconds
  warnings.warn(


  0%|          | 0/30 [00:00<?, ?it/s]

/Users/user/Documents/Magistracy/year_project/hr-ai-scout/venv_hr_ai_scout/lib/python3.13/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.000682830810546875 seconds
  warnings.warn(


  0%|          | 0/30 [00:00<?, ?it/s]

als_score в train (нули): 18
als_score в test  (нули): 0


## 7. Metrics

In [21]:
def calculate_metrics(df_test: pd.DataFrame) -> pd.DataFrame:
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    vacancy_ids = df_test['vacancy_id'].unique()
    
    for vacancy_id in vacancy_ids:
        mask = df_test['vacancy_id'] == vacancy_id
        y_true = df_test.loc[mask, 'target'].values
        y_score = df_test.loc[mask, 'y_pred_proba'].values
        
        if len(y_true) <= 1:
            continue
        
        y_true_2d = y_true.reshape(1, -1)
        y_score_2d = y_score.reshape(1, -1)
        
        ndcg = ndcg_score(y_true_2d, y_score_2d)
        ndcg_scores.append(ndcg)
        
        y_pred_binary = (y_score >= 0.5).astype(int)
        
        precision = precision_score(y_true, y_pred_binary, zero_division=0)
        recall = recall_score(y_true, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
    
    if ndcg_scores:
        print(f"Средний NDCG: {np.mean(ndcg_scores):.4f}")
        print(f"Средний Precision: {np.mean(precision_scores):.4f}")
        print(f"Средний Recall: {np.mean(recall_scores):.4f}")
        print(f"Средний F1-Score: {np.mean(f1_scores):.4f}")

        return np.mean(ndcg_scores), np.mean(precision_scores), np.mean(recall_scores), np.mean(f1_scores)
    else:
        print("Недостаточно данных для расчета метрик")

        return None, None, None, None

## 8. LightGBM + ALS + BERT Similarity

Для каждой BERT-модели запускаем Optuna (15 trials) и логируем NDCG в MLflow.

In [22]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

STUDY_DB_NAME = "sqlite:///local.study.db"


In [23]:
def run_cat_bert(model_name, sim_col):
    short = model_name.split('/')[-1].replace('-', '_').lower()
    cat_bert = categorical_features

    X_tr = X_train[BASE_FEATURES + ['als_score']].copy()
    X_tr[sim_col] = df.loc[X_train.index, sim_col].values

    X_te = X_test[BASE_FEATURES + ['als_score']].copy()
    X_te[sim_col] = df.loc[X_test.index, sim_col].values

    def objective_catboost(trial: optuna.Trial) -> float:
        params = {
            'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
            'model__depth': trial.suggest_int('depth', 3, 10),
            'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
        }
        
        pipeline_catboost = Pipeline([
            ('preprocessing', ColumnTransformer([
                ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
            ], remainder='passthrough')),
            ('model', CatBoostClassifier(
                random_state=RANDOM_STATE, 
                verbose=0, 
                auto_class_weights='Balanced'
            ))
        ])
        
        pipeline_catboost.set_params(**params)
        
        kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        
        for train_idx, val_idx in kfold.split(X_tr, y_train):
            X_fold_train, X_fold_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            pipeline_catboost.fit(X_fold_train, y_fold_train)
            y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
            
            df_val = df.loc[X_fold_val.index].copy()
            df_val['y_pred_proba'] = y_pred_proba[:, 1]
            
            ndcg, _, _, _ = calculate_metrics(df_val)
            
            trial.set_user_attr('pipeline_params', params)
        
        return ndcg

    RUN_NAME_OPTUNE_CATBOOST = f'CatBoostClassifier_optuna_als_{short}'

    with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
        run_id_catboost = run.info.run_id

    STUDY_NAME_CATBOOST = f'CatBoostClassifier_optuna_als_{short}'

    mlflc_catboost = MLflowCallback(
        tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
        metric_name="NDCG",
        create_experiment=False,
        mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
    )

    study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=False)

    study_catboost.optimize(objective_catboost, 
                            n_trials=10, 
                            callbacks=[mlflc_catboost]
                           )
    
    best_params_catboost = study_catboost.best_params
    print(f"Number of finished trials: {len(study_catboost.trials)}")
    print(f"Best params CatBoost: {best_params_catboost}")

    pipeline_catboost_best = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
    pipeline_catboost_best.fit(X_tr, y_train)
    
    y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_te)
    
    df_test_catboost = df.loc[X_te.index].copy()
    df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]
    
    ndcg_catboost_als, precision_catboost_als, recall_catboost_als, f1_catboost_als = calculate_metrics(df_test_catboost)
    metrics_catboost_als = {
        'NDCG': ndcg_catboost_als,
        'Precision': precision_catboost_als,
        'Recall': recall_catboost_als,
        'F1': f1_catboost_als
    }

    RUN_NAME_CATBOOST = f"best_optuna_catboost_als_{short}"
    REGISTRY_MODEL_NAME_CATBOOST = f"best_optuna_catboost_als_{short}"
    
    signature = mlflow.models.infer_signature(X_te, y_test)
    input_example = X_te[:10]
    code_paths = ["BERT_Experiments.ipynb"]
    
    with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
        run_id = run.info.run_id
        
        catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                                 artifact_path=f'best_optuna_catboost_als_{short}',
                                                 registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                                 input_example=input_example,
                                                 code_paths=code_paths,
                                                 await_registration_for=60
                                                )
        mlflow.log_metrics(metrics_catboost_als)
        mlflow.log_params(best_params_catboost)

    return {'Model': model_name, 'sim_col': sim_col, 'Pipeline': pipeline_catboost_best, **metrics_catboost_als}


In [24]:
all_results = []

for model_name, _, _, _ in BERT_MODELS:
    sim_col = bert_sim_cols[model_name]
    print(f"\n{'='*60}\nЭксперимент: {model_name}\n{'='*60}")
    try:
        result = run_cat_bert(model_name, sim_col)
        all_results.append(result)
    except Exception as e:
        print(f"[ОШИБКА] {e}")
        all_results.append({'Model': model_name, 'sim_col': sim_col,
                             'NDCG': None, 'Pipeline': None})
        break



Эксперимент: cointegrated/LaBSE-en-ru
🏃 View run CatBoostClassifier_optuna_als_labse_en_ru at: http://127.0.0.1:5000/#/experiments/2/runs/8d6d3cdcabaa4b5891264d169e81591e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


/var/folders/sl/jz4jkd9n7gd9kb0mz4kk4vp00000gn/T/ipykernel_9749/1349913683.py:57: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc_catboost = MLflowCallback(
[I 2026-05-08 19:39:08,701] A new study created in RDB with name: CatBoostClassifier_optuna_als_labse_en_ru


Средний NDCG: 0.8569
Средний Precision: 0.7467
Средний Recall: 0.8141
Средний F1-Score: 0.7628
Средний NDCG: 0.8607
Средний Precision: 0.7501
Средний Recall: 0.8175
Средний F1-Score: 0.7665


[I 2026-05-08 19:39:35,697] Trial 0 finished with value: 0.8519899245415371 and parameters: {'iterations': 450, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 3.968793330444372}. Best is trial 0 with value: 0.8519899245415371.


Средний NDCG: 0.8520
Средний Precision: 0.7439
Средний Recall: 0.8078
Средний F1-Score: 0.7584
🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/2/runs/89e70c4ff6c14859982ad3d0fda07e47
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Средний NDCG: 0.8543
Средний Precision: 0.6386
Средний Recall: 0.8305
Средний F1-Score: 0.6983
Средний NDCG: 0.8580
Средний Precision: 0.6359
Средний Recall: 0.8337
Средний F1-Score: 0.6972


...

[I 2026-05-08 19:49:28,725] Trial 9 finished with value: 0.8504838236168566 and parameters: {'iterations': 350, 'depth': 3, 'learning_rate': 0.1024932221692416, 'l2_leaf_reg': 2.7551959649510764}. Best is trial 6 with value: 0.8513928169969692.


Средний NDCG: 0.8505
Средний Precision: 0.6552
Средний Recall: 0.8326
Средний F1-Score: 0.7106
🏃 View run 9 at: http://127.0.0.1:5000/#/experiments/2/runs/47956c8881274635a603e66c0e770d97
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Number of finished trials: 10
Best params CatBoost: {'iterations': 500, 'depth': 9, 'learning_rate': 0.019721610970574007, 'l2_leaf_reg': 3.2676417657817622}
Средний NDCG: 0.7541
Средний Precision: 0.5819
Средний Recall: 0.7198
Средний F1-Score: 0.6221


/Users/user/Documents/Magistracy/year_project/hr-ai-scout/venv_hr_ai_scout/lib/python3.13/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/user/Documents/Magistracy/year_project/hr-ai-scout/venv_hr_ai_scout/lib/python3.13/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer c

🏃 View run best_optuna_catboost_als_multilingual_e5_base at: http://127.0.0.1:5000/#/experiments/2/runs/42f0ab40d8be405bbf6b9627f110ff07
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


Created version '1' of model 'best_optuna_catboost_als_multilingual_e5_base'.


## 9. Результаты

In [25]:
NDCG_TFIDF_BASELINE = 0.7816

results_df = pd.DataFrame([
    {'Model': r['Model'], 'NDCG': r['NDCG'],
     'Precision': r.get('Precision'), 'Recall': r.get('Recall'), 'F1': r.get('F1')}
    for r in all_results
])
results_df['Delta vs TF-IDF'] = results_df['NDCG'] - NDCG_TFIDF_BASELINE
results_df = results_df.sort_values('NDCG', ascending=False).reset_index(drop=True)

print(f"Базовая линия TF-IDF: NDCG = {NDCG_TFIDF_BASELINE}")
print()
print(results_df[['Model','NDCG','Delta vs TF-IDF']].to_string(index=False))


Базовая линия TF-IDF: NDCG = 0.7816

                        Model     NDCG  Delta vs TF-IDF
     cointegrated/LaBSE-en-ru 0.754102        -0.027498
intfloat/multilingual-e5-base 0.754074        -0.027526
ai-forever/sbert_large_nlu_ru 0.748948        -0.032652


In [26]:
# valid = [r for r in all_results if r.get('NDCG') is not None]
# if valid:
#     best = max(valid, key=lambda r: r['NDCG'])
#     if best['NDCG'] > NDCG_TFIDF_BASELINE:
#         fname = f"pipeline_xgb_als_{best['sim_col']}.pkl"
#         with open(fname, 'wb') as f:
#             pickle.dump(best['Pipeline'], f)
#         print(f"Лучший пайплайн сохранён: {fname}")
#         print(f"NDCG: {best['NDCG']:.4f}  (TF-IDF: {NDCG_TFIDF_BASELINE:.4f})")
#     else:
#         print(f"TF-IDF остаётся лучшим ({NDCG_TFIDF_BASELINE:.4f}).")
#         print(f"Лучший BERT: {best['Model']}  NDCG={best['NDCG']:.4f}")
